In [193]:
#load in packages 
import numpy as np
import pandas as pd

#regex
import re

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import mutual_info_classif

from nltk.stem import SnowballStemmer
from nltk.tokenize import word_tokenize

import matplotlib.pyplot as plt


In [214]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import nltk
import matplotlib.pyplot as plt
from nltk.corpus import words

import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.corpus import words

In [215]:
nltk.download('wordnet')
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\jarem\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jarem\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jarem\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [216]:
import nltk

nltk.download('punkt')        # For word_tokenize
nltk.download('stopwords')    # For stopwords
nltk.download('wordnet')      # For WordNetLemmatizer
nltk.download('words')        # For nltk.corpus.words


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\jarem\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jarem\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\jarem\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package words to
[nltk_data]     C:\Users\jarem\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!


True

In [217]:
items_df = pd.read_csv("categorized_items3.csv").iloc[:, 2:]
items_df.dropna()
items_df["processed"] = items_df["processed"].astype(str)

In [235]:
items_df.head(1)

,store_id,product_id,product_title,summary,description,TSD,processed,breadcrumb_clean,category
0,2,1111071,"Sainsbury's Thick Bleach, Lemon 2L",Thick Bleach Citrus,"Thick Bleach Citrus\nwith limescale deterrent\nThick Bleach Citrus\n*Kills 99.9% of bacteria including Listeria, Salmonella and E. coli, and flu viruses when used neat. Kills Coronavirus (tested against surrogate coronavirus following EN14476).\nLeaves a fresh perfume, removes stains.\nHelps prevent limescale formation.\nSafe for use with septic tanks.","Sainsbury's Thick Bleach, Lemon 2L Thick Bleach Citrus Thick Bleach Citrus\nwith limescale deterrent\nThick Bleach Citrus\n*Kills 99.9% of bacteria including Listeria, Salmonella and E. coli, and flu viruses when used neat. Kills Coronavirus (tested against surrogate coronavirus following EN14476).\nLeaves a fresh perfume, removes stains.\nHelps prevent limescale formation.\nSafe for use with septic tanks.",thick bleach lemon l thick bleach citrus thick bleach citrus deterrent thick bleach citrus kill bacteria salmonella e coli flu virus used neat kill tested surrogate following en leaf fresh perfume remove stain help prevent formation safe use septic tank,Groceries | Household | Household essentials | Sainsburys Thick Bleach | Lemon 2L,Cleaning Liquid


In [219]:
items_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5970 entries, 0 to 5969
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   store_id          5970 non-null   int64 
 1   product_id        5970 non-null   int64 
 2   product_title     5970 non-null   object
 3   summary           5703 non-null   object
 4   description       4760 non-null   object
 5   TSD               4665 non-null   object
 6   processed         5970 non-null   object
 7   breadcrumb_clean  5970 non-null   object
 8   category          5970 non-null   object
dtypes: int64(2), object(7)
memory usage: 419.9+ KB


## Create union for breadcrumb and processed and lemmatise

In [231]:
stop_words = stopwords.words('english')
english_words = set(words.words())

In [237]:
df = items_df.copy()
df['text'] = df['processed'].astype(str) + ' ' + df['breadcrumb_clean'].astype(str)

In [238]:
# Preprocessing function
def preprocess_text(text):
    text = text.lower() # Lowercase
    text = re.sub(r'[^a-z\s]', '', text) # Remove punctuation
    tokens = word_tokenize(text) # Tokenize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens] # Lemmatize
    tokens = [word for word in tokens if word not in stopwords.words('english')] # Remove stopwords
    tokens = [word for word in tokens if word in english_words] # Remove non-english words
    return " ".join(tokens)

# Apply preprocessing
df['text'] = df['text'].apply(preprocess_text)

LookupError: 
**********************************************************************
  Resource [93mpunkt_tab[0m not found.
  Please use the NLTK Downloader to obtain the resource:

  [31m>>> import nltk
  >>> nltk.download('punkt_tab')
  [0m
  For more information see: https://www.nltk.org/data.html

  Attempted to load [93mtokenizers/punkt_tab/english/[0m

  Searched in:
    - 'C:\\Users\\jarem/nltk_data'
    - 'c:\\Users\\jarem\\AppData\\Local\\Programs\\Python\\Python313\\nltk_data'
    - 'c:\\Users\\jarem\\AppData\\Local\\Programs\\Python\\Python313\\share\\nltk_data'
    - 'c:\\Users\\jarem\\AppData\\Local\\Programs\\Python\\Python313\\lib\\nltk_data'
    - 'C:\\Users\\jarem\\AppData\\Roaming\\nltk_data'
    - 'C:\\nltk_data'
    - 'D:\\nltk_data'
    - 'E:\\nltk_data'
**********************************************************************


## Assign gender

In [227]:
# Function to classify gender based on the description

def classify_gender(descr):
    # Define regex patterns for male, female, and neutral categories 
    male_pattern = r'\b(male|men|mens|man|mans|masculine|him|for him|his)\b'
    female_pattern = r'\b(feminine|for her|her|hers|women|female|womans|womens|woman|makeup|make up|nailpolish|nail polish)\b' # add makeup and nailpolish as products marketed for women
    
    # Check if the breadcrumb matches the male pattern
    if re.search(male_pattern, descr, re.IGNORECASE):
        return 'male'
    # Check if the breadcrumb matches the female pattern
    elif re.search(female_pattern, descr, re.IGNORECASE):
        return 'female'
    # Assign neutral if neither male nor female is found
    else:
        return 'neutral'

In [228]:
gender = items_df['processed'].apply(classify_gender)

In [173]:
# Apply the function to classify gender
df['gender'] = df['processed'].apply(classify_gender)

# Create binary dummy columns
df['is_male'] = (df['gender'] == 'male').astype(int)
df['is_female'] = (df['gender'] == 'female').astype(int)
df['is_neutral'] = (df['gender'] == 'neutral').astype(int)

In [178]:
df.drop(columns=["TSD","product_title","summary", "description", "gender"], inplace=True)

In [179]:
df.index=df["product_id"]

In [180]:
y = df[["is_male","is_female","is_neutral"]]

In [181]:
y

,is_male,is_female,is_neutral
product_id,,,
1111071,0,0,1
2035727,0,0,1
2061504,0,0,1
2086910,0,0,1
2192529,0,0,1
...,...,...,...
502050631293631294,0,0,1
922160825648825649,0,0,1
895947729850729851,0,0,1


In [182]:
pd.set_option('display.max_colwidth', None)

In [183]:
df.head()

,store_id,product_id,processed,breadcrumb_clean,category,is_male,is_female,is_neutral
product_id,,,,,,,,
1111071,2,1111071,thick bleach lemon l thick bleach citrus thick bleach citrus deterrent thick bleach citrus kill bacteria salmonella e coli flu virus used neat kill tested surrogate following en leaf fresh perfume remove stain help prevent formation safe use septic tank,Groceries | Household | Household essentials | Sainsburys Thick Bleach | Lemon 2L,Cleaning Liquid,0,0,1
2035727,2,2035727,toilet bleach germ free original l original thick bleach clean surface around home original thick bleach clean surface around home unstoppable battle germ bleach bacteria virus used toilet cleaner well cleaner surface around home diluted toilet cleaner help prevent buildup make white surface lasting freshness bleach brand easy way protect ensure safety one pair disinfectant toilet block maximum cleaning power long lasting freshness go battle way germ hide whether bathroom anywhere else house original thick bleach bacteria virus leaving surface help keep family safe active molecule stick even waterline helping prevent buildup better still thick bleach make white surface lasting freshness diluted used cleaner around home bathroom cleaner even drain cleaner disinfectant bleach brand also suitable use home septic tank ha protecting family since today protect million family full range cleaning product home whilst also fighting poor sanitation globally ha million people gain access clean safe toilet thousand child clean water toilet facility school thereby helping improve attendance educational achievement bacteria virus like vaccinia virus use safely always read label product information use always follow instruction dilution source data value unit sale household cleaning toilet bleach client defined,Groceries | Household | Household essentials | Domestos Toilet Bleach Germ Free Original 2L,Cleaning Liquid,0,0,1
2061504,2,2061504,cavity protection pump enamel strengthening enamel strengthening great mint taste breath dual system fluoride arginine advanced cavity protection bacteria protect teeth defend smile maximum cavity protection give teeth best protection unique system fluoride natural arginine cavity protection v previous formula give smile best protection maximum cavity protection new best ever formula cavity protection dual system unique combination fluoride natural arginine give x enamel effectively caries give advanced cavity protection also ha superior technology bacteria chance weaken enamel great mint flavour breath even reason defend smile cavity protection cavity protection v previous formulation week mineral change v regular fluoride v regular fluoride,Groceries | Toiletries & health | Toiletries & health essentials | Colgate Cavity Protection Toothpaste Pump 100ml,Oral Care,0,0,1
2086910,2,2086910,face body hand intensive protective care intensive protective care even dry area skin compatibility original whole family skin protective care need stay soft supple ideal daily use wherever skin need care care skin skin compatibility,Groceries | Beauty & cosmetics | Bath & body | Nivea Creme Moisturiser for Face Body & Hands 50ml,Skincare,0,0,1
2192529,2,2192529,face body hand intensive protective care intensive protective care even dry area skin compatibility original whole family skin protective care need stay soft supple ideal daily use wherever skin need care care skin skin compatibility,Groceries | Beauty & cosmetics | Bath & body | Nivea Creme Moisturiser for Face Body & Hands 200ml,Skincare,0,0,1


In [152]:
overlapping_rows = ((df[['is_male', 'is_female', 'is_neutral']].sum(axis=1)) > 1).sum()
summary_stats = df[['is_male', 'is_female', 'is_neutral']].sum().reset_index()
summary_stats.columns = ['gender', 'number of classifications']

display(summary_stats)

display(f"Number of rows with multiple binary columns set to 1: {overlapping_rows}")

,gender,number of classifications
0,is_male,695
1,is_female,431
2,is_neutral,4844


'Number of rows with multiple binary columns set to 1: 0'

## CountVecotrisation + logistic regression

In [202]:


# 1. Define your regex keyword sets (the ones you already used to classify gender)
male_keywords = {'male', 'men', 'mens', 'man', 'mans', 'masculine', 'him', 'for him', 'his', 'boy','boys','testosterone'}
female_keywords = {'feminine', 'for her', 'her', 'hers', 'women', 'female', 'womans', 'womens', 'woman', 'makeup', 'make up', 'nailpolish', 'girl','girls', 'nail polish', 'pregnancy','conception','menopause', 'pregnant'}
regex_keywords = male_keywords.union(female_keywords)

# 2. Prepare your text data (ASK SAM TO CLEAN BREADCRUMB TOO
texts = df['processed'].fillna("").values

# 3. Vectorize the text data.
#    You might want to remove stopwords and work with lowercase tokens.
vectorizer = CountVectorizer(lowercase=True, stop_words='english', max_df=0.99, min_df=0.01)
X = vectorizer.fit_transform(texts)
feature_names = np.array(vectorizer.get_feature_names_out())

# 4. Prepare your target variable.
#    Here we assume that each row has only one gender set to 1.
#    We create a single label ('male', 'female', or 'neutral') for each row.
y = df[['is_male', 'is_female', 'is_neutral']].apply(
    lambda row: 'male' if row['is_male'] == 1 else ('female' if row['is_female'] == 1 else 'neutral'),
    axis=1
)

# 5. Train a classifier.
#    Using Logistic Regression in a one-vs-rest scheme.
clf = LogisticRegression(multi_class='ovr', max_iter=10000)
clf.fit(X, y)

# 6. Extract the discriminating words based on the model’s coefficients.
#    For each class, we sort the coefficients (features) and select the top N words.
top_words_dict = {}
top_n = 100  # number of top words to consider per class

for i, cls in enumerate(clf.classes_):
    coef = clf.coef_[i]
    sorted_indices = coef.argsort()[::-1]  # indices sorted descending by weight
    # Filter out any words that appear in the regex keywords list
    top_words = [feature_names[idx] for idx in sorted_indices if feature_names[idx] not in regex_keywords]
    top_words_dict[cls] = top_words[:top_n]

# 7. Ensure exclusivity:
#    For each gender class, remove words that also appear among the top words of any other class.
exclusive_words = {}
for cls in top_words_dict:
    exclusive = []
    for word in top_words_dict[cls]:
        # Check if this word appears in any other class's top words
        appears_elsewhere = any(word in top_words_dict[other_cls] for other_cls in top_words_dict if other_cls != cls)
        if not appears_elsewhere:
            exclusive.append(word)
    exclusive_words[cls] = exclusive

# Display the resulting exclusive discriminating words per gender
print("Discriminating words per gender (exclusive):")
for gender, words in exclusive_words.items():
    print(f"{gender}: {words}")


Discriminating words per gender (exclusive):
female: ['hormonal', 'floral', 'nutrition', 'additional', 'activity', 'nutrient', 'contribute', 'people', 'vision', 'vegetarian', 'brain', 'chewy', 'preservative', 'liquid', 'ultimate', 'pink', 'fragrance', 'nervous', 'available', 'quickly', 'sweet', 'selenium', 'child', 'close', 'intended', 'mental', 'size', 'rash', 'dose', 'maximum', 'supplement', 'gluten', 'cotton', 'taken', 'feel', 'delicate', 'advanced', 'global', 'aid', 'confident', 'cool', 'level', 'stress', 'acid', 'expert', 'danger', 'registered', 'pure', 'bar', 'salon', 'lightweight', 'lasting', 'description', 'smoother', 'pantothenic', 'proven', 'cell', 'control', 'aloe', 'avoid', 'possible', 'plastic', 'whilst', 'lemon', 'neutral', 'glass', 'eu', 'convenient', 'unit', 'tiredness', 'step', 'medicine', 'sweat', 'orange', 'sure', 'source', 'excellent', 'keeping', 'luxurious', 'spray', 'harmful', 'follow', 'person', 'developer', 'diet', 'consult']
male: ['sharp', 'note', 'zinc', 'gam

c:\Users\jarem\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


#

# TF-IDF Vectorization + Mutual Information

In [201]:
y = df[['is_male', 'is_female', 'is_neutral']].apply(
    lambda row: 'male' if row['is_male'] == 1 else ('female' if row['is_female'] == 1 else 'neutral'),
    axis=1
)

# Convert y to categorical (if needed)
y = pd.Categorical(y)

from sklearn.preprocessing import Binarizer

# Apply binarization to the TF-IDF matrix
binarizer = Binarizer()
X_binarized = binarizer.fit_transform(X_tfidf)

# 2. Vectorize the text data using TF-IDF
vectorizer = TfidfVectorizer(lowercase=True, stop_words='english', max_features=5000)
X_tfidf = vectorizer.fit_transform(df['processed'].fillna(""))

# 3. Compute Mutual Information scores
mi_scores = mutual_info_classif(X_tfidf, y, discrete_features=False)

# 4. Extract the top words for each gender
feature_names = np.array(vectorizer.get_feature_names_out())

# Create a dictionary to store top words per gender
top_words_dict = {}
n_top_words = 20  # Number of top words to extract

# Compute MI scores separately for each class
for gender in ['male', 'female', 'neutral']:
    # Mask for current gender
    mask = (y == gender).values

    # MI scores specific to the current gender
    mi_scores_gender = mutual_info_classif(X_tfidf[mask], [gender] * sum(mask), discrete_features=False)
    
    # Sort and extract top N words
    top_indices = np.argsort(mi_scores_gender)[::-1][:n_top_words]
    top_words = feature_names[top_indices]
    
    # Store the result
    top_words_dict[gender] = top_words.tolist()

# Display the results
print("\n🔥 Top Discriminating Words by Gender (TF-IDF + Mutual Information):")
for gender, words in top_words_dict.items():
    print(f"{gender}: {words}")

ValueError: Sparse matrix `X` can't have continuous features.

## Naive Bayes + Log Probabilities

In [199]:
from sklearn.naive_bayes import MultinomialNB

# 1. Fit a Multinomial Naive Bayes model
mnb = MultinomialNB()
mnb.fit(X_tfidf, y)

# 2. Extract log probabilities
#    Each row corresponds to a class (male, female, neutral)
log_probs = mnb.feature_log_prob_

# 3. Map log probabilities to gender labels
top_words_dict_nb = {}
n_top_words = 20  # Number of top words

# Extracting top discriminating words for each gender
for idx, gender in enumerate(mnb.classes_):
    # Sort features by log probability
    top_indices = np.argsort(log_probs[idx])[::-1][:n_top_words]
    top_words = feature_names[top_indices]
    
    # Store the result
    top_words_dict_nb[gender] = top_words.tolist()

# Display the results
print("\n🔥 Top Discriminating Words by Gender (Naive Bayes Log Probabilities):")
for gender, words in top_words_dict_nb.items():
    print(f"{gender}: {words}")


IndexError: index 4927 is out of bounds for axis 0 with size 1031

In [186]:
female_ratio = sum(gender == "female") / len(gender)
male_ratio = sum(gender == "male") / len(gender)
neutral_ratio = sum(gender == "neutral") / len(gender)

print(f"Classified as female: {female_ratio:.2%}")
print(f"Classified as male: {male_ratio:.2%}")
print(f"Classified as neutral: {neutral_ratio:.2%}")

TypeError: 'bool' object is not iterable

In [104]:
neutrals = df[(gender == "neutral") & (df["processed"] != "nan")]["processed"]
# neutrals = neutrals.drop(["product_id", "store_id", "gender_breadcrumb"], axis=1)
neutrals.info()

<class 'pandas.core.series.Series'>
Index: 3539 entries, 0 to 5964
Series name: processed
Non-Null Count  Dtype 
--------------  ----- 
3539 non-null   object
dtypes: object(1)
memory usage: 55.3+ KB


In [30]:
females = df[(gender == "female") & (df["processed"] != "nan")]["processed"]
# neutrals = neutrals.drop(["product_id", "store_id", "gender_breadcrumb"], axis=1)
females.info()

<class 'pandas.core.series.Series'>
Index: 431 entries, 17 to 5888
Series name: processed
Non-Null Count  Dtype 
--------------  ----- 
431 non-null    object
dtypes: object(1)
memory usage: 6.7+ KB


In [31]:
males = df[(gender == "male") & (df["processed"] != "nan")]["processed"]
# neutrals = neutrals.drop(["product_id", "store_id", "gender_breadcrumb"], axis=1)
males.info()

<class 'pandas.core.series.Series'>
Index: 695 entries, 48 to 5954
Series name: processed
Non-Null Count  Dtype 
--------------  ----- 
695 non-null    object
dtypes: object(1)
memory usage: 10.9+ KB


In [76]:
vect = CountVectorizer(ngram_range=(1, 1),
                              stop_words='english',
                              lowercase=True)
                              #max_df=0.90, 
                              #min_df=0.1)

In [77]:
vect_norm = TfidfVectorizer(ngram_range=(1, 1), #
                              stop_words='english',
                              lowercase=True,
                              max_df=0.90, 
                              min_df=0.1)

## Males

In [ ]:
males_vect = vect_norm.fit_transform(males)

# create DFM
males_dfm = pd.DataFrame(
    males_vect.toarray(),
    columns=vect_norm.get_feature_names_out()
)

In [88]:
males_dfm

,accessory,accurate,additional,adult,advice,aerosol,aim,alcohol,allergen,allergy,...,wash,water,way,welcome,whats,whatsoever,woman,work,world,zinc
0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.086215,0.086215
1,0.0,0.0,0.0,0.063945,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.092668,0.0,0.106128,0.000000
2,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.122608,0.150269,0.0,0.0,0.054843,0.0,0.044670,0.0,0.000000,0.000000
3,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.122541,0.150187,0.0,0.0,0.054812,0.0,0.044646,0.0,0.000000,0.000000
4,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.179391,0.131917,0.0,0.0,0.048145,0.0,0.039215,0.0,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
690,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000
691,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.105640
692,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000
693,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000


In [ ]:
male_common_words = ((males_dfm > 0).sum() / len(males_dfm))
male_common_words = pd.DataFrame(male_common_words, columns=["frequencies"])
male_common_words.index.name = 'words'
display(male_common_words.sort_values(by="frequencies", ascending=False).head(30))

,frequencies
words,
use,0.631655
skin,0.621583
day,0.618705
ha,0.592806
need,0.576978
designed,0.574101
feature,0.569784
product,0.526619
experience,0.520863


## Females

In [93]:
females_vect = vect_norm.fit_transform(females)

# create DFM
females_dfm = pd.DataFrame(
    females_vect.toarray(),
    columns=vect_norm.get_feature_names_out()
)

In [94]:
females_dfm

,accessory,accurate,acid,additional,adult,advanced,advice,aerosol,alcohol,allergen,...,water,way,week,whatsoever,white,wont,work,world,year,zinc
0,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.000000,0.0,0.0,0.0,...,0.0,0.00000,0.00000,0.0,0.0,0.0,0.067438,0.083214,0.149851,0.000000
1,0.0,0.0,0.323881,0.092613,0.0,0.00000,0.047614,0.0,0.0,0.0,...,0.0,0.00000,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.141297
2,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.000000,0.0,0.0,0.0,...,0.0,0.00000,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
3,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.000000,0.0,0.0,0.0,...,0.0,0.00000,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
4,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.000000,0.0,0.0,0.0,...,0.0,0.00000,0.05469,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
426,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.000000,0.0,0.0,0.0,...,0.0,0.04967,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
427,0.0,0.0,0.000000,0.000000,0.0,0.00000,0.000000,0.0,0.0,0.0,...,0.0,0.00000,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.200487,0.000000
428,0.0,0.0,0.204596,0.000000,0.0,0.07357,0.000000,0.0,0.0,0.0,...,0.0,0.00000,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.148762
429,0.0,0.0,0.387691,0.055430,0.0,0.00000,0.000000,0.0,0.0,0.0,...,0.0,0.00000,0.00000,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.169135


In [95]:
female_common_words = ((females_dfm > 0).sum() / len(females_dfm))
female_common_words = pd.DataFrame(female_common_words, columns=["frequencies"])
female_common_words.index.name = 'words'
display(female_common_words.sort_values(by="frequencies", ascending=False).head(30))

,frequencies
words,
day,0.647332
skin,0.624130
designed,0.582367
use,0.575406
ha,0.556845
help,0.529002
need,0.529002
brand,0.505800
product,0.496520


## Neutrals

In [97]:
neutrals_vect = vect_norm.fit_transform(neutrals)

# create DFM
neutrals_dfm = pd.DataFrame(
    neutrals_vect.toarray(),
    columns=vect_norm.get_feature_names_out()
)

In [99]:
neutral_common_words = ((neutrals_dfm > 0).sum() / len(neutrals_dfm))
neutral_common_words = pd.DataFrame(neutral_common_words, columns=["frequencies"])
neutral_common_words.index.name = 'words'
display(neutral_common_words.sort_values(by="frequencies", ascending=False).head(30))

,frequencies
words,
use,0.537440
product,0.479797
ha,0.470189
care,0.467646
ingredient,0.467364
day,0.457474
help,0.433738
food,0.428370
designed,0.413959


## Use LLM to create a dictionary for ADM

In [ ]:
# Gendered words dictionary for product packaging
gendered_words = {
    "feminine": [
        "gentle", "soft", "delicate", "radiant", "luminous", "soothing", "dewy",
        "floral", "elegance", "grace", "charm", "blossom", "rose", "pearl", "flower" , "nurturing", "nutritious"
        "for her", "designed for women", "feminine touch" , "volume" , "voluminous", "young" 
    ],
    "masculine": [
        "strong", "rugged", "bold", "powerful", "athletic", "robust", "tough",
        "strength", "force", "cedarwood", "musk", "steel", "for him",
        "engineered for men", "masculine scent", "beast", 
    ],
    "neutral": [
        "fresh", "clean", "natural", "eco-friendly", "sustainable", "modern",
        "quality", "care", "innovation", "balance", "nature"
    ]
}


In [ ]:
female_keywords = {
    "female": {
        "food_beverages": [
            "guilt-free", "nourishing", "wholesome", "superfood", "antioxidant-rich",
            "low-calorie", "dairy-free", "revitalizing", "revitilize", "revitilise","revitilising", "skin-boosting", "detoxifying",
            "pure", "farm-to-table", "probiotic", "vitamin-packed", "heart-healthy",
            "indulgent", "crave-worthy", "mood-enhancing", "meal-prep-friendly"
        ],
        "clothing_apparel": [
            "flattering", "figure", "hugging", "chic", "feminine", "tailored",
            "confidence-boosting", "soft-to-the-touch", "curve-loving", "luxe",
            "couture-inspired", "occasion-ready", "wrinkle-resistant", "seasonless",
            "trend-forward", "size-inclusive"
        ],
        "beauty_skincare": [
            "hydrating", 
            "radiant", 
            "rejuvenating", 
            "age-defying", 
            "anti-aging",
            "glow-boosting", 
            "non-comedogenic", 
            "beauty", 
            "paraben-free",
            "sulfate-free", 
            "luminous", 
            "plumping", 
            "firming", 
            "brightening", 
            "plumed", 
            "hug", 
            "moisturizing", 
            "moisture", 
            "glitter", 
            "sensitive", 
            "pore-minimizing", 
            "pore-minimising",
            "spa-like", 
            "silky", 
            "velvety", 
            "velvet",
            "smudge-proof", 
            "pamper", 
            "young", 
            "elegant", 
            "elegance", 
            "pampering", 
            "clinically proven", 
            "sensitive-skin-friendly", 
            "revitalizing", 
            "revitilize", 
            "revitilise",
            "revitilising", 
            "miracle",
            "miracolous", 
            "frizzy",
            "luxuriously",
            "luxurious",
            "luxury",
            "sleek",
            "smoothness",
            "gently",
            "restore",
            "restoring",
            "macadamia",
            "moist",
            "moisturizes",
            "moisturises",
            "gentle", 
            "soft", 
            "delicate", 
            "radiant", 
            "luminous", 
            "soothing", 
            "dewy", 
            "floral", 
            "elegance", 
            "grace", 
            "charm", 
            "blossom", 
            "rose", 
            "pearl", 
            "flower", 
            "sensational",
            "sensual",
            "nurturing", 
            "nutritious",
            "her",
            "hers", 
            "designed for women", 
            "feminine touch" , 
            "volume" , 
            "voluminous", 
            "spoil", 
            "nourishing", 
            "avocado oil", 
            "radiating", 
            "smooth", 
            "coloured", 
            "colour", 
            "coloured-hair", 
            "shine",             
            "vibrancy", 
            "restore", 
            "restoring", 
            "regenerate", 
            "haircare", 
            "care", 
            "skincare", 
            "harmony", 
            "feel", 
            "healthy", 
            "vibrant", 
            "feminine",
            "women",
            "woman",
            "woman's",
            "women's"
        ],
        "cross_category": [
            "empowering", "curated", "personalized", "transformative", "mindful",
            "bespoke", "award-winning", "iconic", "must-have", "essentials",
            "perfect for you", "perfect for everyday"
        ]
    }
}

male_keywords = {
    "male": {
        "food_beverages": [
            "muscle-fuel", "keto-friendly", "low-carb", "energy-packed", "grass-fed",
            "no-added-sugar", "hearty", "bold flavors", "crafted", "smoked",
            "slow-cooked", "testosterone-supporting", "crunchy", "grillable",
            "fuel-for-the-day"
        ],
        "clothing_apparel": [
            "rugged", "moisture-wicking", "quick-dry", "athletic-fit", "slim-fit",
            "classic-cut", "performance-ready", "stretch-flex", "reinforced",
            "heavy-duty", "weather-resistant", "odour-resistant", "anti-microbial",
            "workwear-inspired", "built-to-last", "multi-pocket", "tech-friendly"
        ],
        "beauty_grooming": [
            "barber-approved", 
            "beard-friendly", 
            "anti-itch", 
            "long-lasting hold",
            "oil-free", 
            "matte finish", 
            "exfoliating", 
            "charcoal-infused",
            "post-shave", 
            "fade-resistant", "strengthening", 
            "travel-sized",
            "tattoo-safe", 
            "sweat-proof", 
            "protect",
            "ice",
            "iced",
            "zesty",
            "citrus",
            "chest",
            "irresistible",
            "sweat",
            "fight",
            "kill",
            "men's",
            "man's",
            "bold", 
            "no-nonsense", 
            "masculine", 
            "streamlined", 
            "modern",
            "go-anywhere", 
            "built-tough", 
            "precision-engineered",
            "mission-critical", 
            "uncompromising", 
            "lean",
            "attack",
            "strengthen",
            "stronger",
            "strongest",
            "bulldog",
            "technology",
            "odour-control",
            "odor-control",
            "masculine",
            "balanced",
            "engineered",
            "engineering",
            "confidence",
            "odor",
            "odour",
            "his",
            "him",
            "he",
            "men",
            "male",
            "man",
            "father",
            "son",
            "yourself",
            "be yourself",
            
            
            
        ]
    }
}

neutral_keywords = {
    "neutral": {
        "food_beverages": [
            "sustainably sourced", "organic", "plant-based", "non-GMO", "low-waste",
            "nutrient-dense", "ethically produced", "allergy-friendly", "gluten-free",
            "vegan", "fair-trade", "minimally processed", "whole-food", "balanced",
            "energy-boosting", "immune-supporting", "hydrating", "seasonal",
            "locally grown", "recyclable packaging", "zero-added-sugar", "flexitarian",
            "climate-friendly", "compostable", "mindfully crafted"
        ],
        "clothing_apparel": [
            "unisex", "size-inclusive", "eco-friendly", "versatile", "durable",
            "breathable", "easy-care", "timeless", "minimalist", "adjustable",
            "lightweight", "recycled materials", "modular", "gender-neutral fit",
            "functional", "adaptive", "travel-ready", "certified sustainable",
            "biodegradable", "packable", "long-lasting"
        ],
        "beauty_personal_care": [
            "gentle", 
            "hypoallergenic", 
            "fragrance-free", 
            "cruelty-free", 
            "reef-safe",
            "non-toxic", 
            "pH-balanced", 
            "zero-waste", 
            "dermatologist-tested",
            "sensitive-skin safe", 
            "refillable", 
            "waterless", 
            "carbon-neutral",
            "all-in-one", 
            "travel-friendly", 
            "plastic-free", 
            "universal",
            "transparent ingredients", 
            "ethically sourced", 
            "compostable packaging", 
            "clean",

            
            
        ],
        "cross_category": [
            "inclusive", "sustainable", "ethical", "innovative", "accessible",
            "adaptable", "transparent", "holistic", "universal", "eco-conscious",
            "future-proof", "community-driven", "thoughtfully designed",
            "quality-crafted", "multi-functional", "user-friendly", "planet-positive",
            "socially responsible"
        ]
    }
}

__Potential challenges__

1. How do we standardize prices to put it on the same scale?

    * option 1: create a categorical variable capturing the type of pricing -- but then how do you convert it into standardized df?

    * option 2 :bring to common denominator as much as possible


2. How do we study the gender price gap?

    * we need to find a way to assign the products into a category (female/male/neither)

3. What method do we study this with?

    * Moshary et al. 2023 use regression


__Pre-processing. Things to do__

- transform data to datetime object


Use one store info to categorize into female/male (Cyril's idea)